# Estatistica, Probabilidade e Previsao

## Live 3 - Applied Math for Data Science

Nesta aula, vamos comparar dois jeitos de prever o futuro em series temporais:

- **Regressao linear** para capturar tendencia global
- **ARIMA** para capturar dependencia temporal

O caso da aula sera a previsao de usuarios ativos mensais de uma plataforma de IA.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
from statsmodels.tsa.arima.model import ARIMA

plt.style.use('seaborn-v0_8-darkgrid')
np.set_printoptions(precision=4, suppress=True)
warnings.filterwarnings('ignore')

print('Ambiente pronto para previsao temporal.')

---

## 1. Criando uma serie temporal com tendencia, sazonalidade e ruido

Vamos simular 72 meses de atividade de uma plataforma. A serie tem:

- crescimento estrutural
- oscilacao sazonal ao longo do ano
- ruido aleatorio

In [ ]:
rng = np.random.default_rng(7)
meses = np.arange(72)

tendencia = 200 + 4.5 * meses
sazonalidade = 28 * np.sin(2 * np.pi * meses / 12)
ruido = rng.normal(0, 8, size=len(meses))

serie = tendencia + sazonalidade + ruido

print(f'Tamanho da serie: {len(serie)} meses')
print(f'Media: {np.mean(serie):.2f}')
print(f'Desvio padrao: {np.std(serie):.2f}')

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(meses, serie, color='#74c7ec', linewidth=2.5, label='Usuarios ativos mensais')
plt.title('Serie temporal simulada')
plt.xlabel('Mes')
plt.ylabel('Usuarios ativos (milhares)')
plt.legend()
plt.show()

### Leitura estatistica rapida

Em previsao, a estatistica entra primeiro para responder:

- ha crescimento?
- ha repeticao sazonal?
- o ruido parece pequeno ou grande?

Sem essa leitura, escolher um modelo vira tentativa e erro.

### Exemplos de vida real para previsao

Forecast aparece em muitos produtos e operacoes reais:

- **E-commerce**: prever demanda futura para ajustar estoque
- **Fintech**: prever transacoes por hora para evitar sobrecarga
- **Mobilidade**: prever corridas ou trafego em horarios de pico
- **Saude**: prever ocupacao de leitos e consumo de insumos
- **Plataformas de IA**: prever uso de GPU e numero de requisicoes

Nesta aula, vamos usar o ultimo caso: crescimento de usuarios ativos mensais de uma plataforma de IA.

---

## 2. Separando treino e teste

Vamos treinar com os primeiros 60 meses e testar nos 12 meses finais.

In [ ]:
train_size = 60
x_train = meses[:train_size]
y_train = serie[:train_size]
x_test = meses[train_size:]
y_test = serie[train_size:]

print(f'Treino: {len(y_train)} pontos')
print(f'Teste: {len(y_test)} pontos')

---

## 3. Antes do ARIMA: comecando pelo ARMA

Antes de falar de ARIMA, vale separar as pecas:

- **AR (AutoRegressive)**: o valor atual depende de valores passados da propria serie
- **MA (Moving Average)**: o valor atual depende dos erros passados
- **ARMA**: combina os dois, assumindo uma serie mais estavel

Esquematicamente:

$$y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + \dots + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \theta_2 \varepsilon_{t-2} + \dots$$

### O que o I resolve

ARMA puro sofre quando a serie tem tendencia forte. O `I` de ARIMA vem de **Integrated** e, na pratica, significa diferenciar a serie:

$$\Delta y_t = y_t - y_{t-1}$$

Se uma diferenca nao bastar, podemos aplicar de novo:

$$\Delta^2 y_t = \Delta y_t - \Delta y_{t-1}$$

In [ ]:
primeira_diferenca = np.diff(serie, n=1)
segunda_diferenca = np.diff(serie, n=2)

fig, axes = plt.subplots(3, 1, figsize=(12, 8))

axes[0].plot(meses, serie, color='#74c7ec', linewidth=2.2)
axes[0].set_title('Serie original')

axes[1].plot(meses[1:], primeira_diferenca, color='#8ce99a', linewidth=2.2)
axes[1].axhline(0, color='white', linestyle=':', alpha=0.6)
axes[1].set_title('Primeira diferenca: y_t - y_(t-1)')

axes[2].plot(meses[2:], segunda_diferenca, color='#ff8fab', linewidth=2.2)
axes[2].axhline(0, color='white', linestyle=':', alpha=0.6)
axes[2].set_title('Segunda diferenca: Delta y_t - Delta y_(t-1)')
axes[2].set_xlabel('Mes')

plt.tight_layout()
plt.show()

### Como o MA funciona

O `MA` nao e uma media movel simples dos dados observados. Ele usa os **erros passados** para corrigir a previsao atual:

$$y_t = c + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \theta_2 \varepsilon_{t-2} + \dots$$

Se o modelo errou para cima ou para baixo recentemente, isso pode carregar informacao util para o instante atual.

In [ ]:
coef_base = np.polyfit(x_train, y_train, deg=1)
base_train = coef_base[0] * x_train + coef_base[1]
residuos_base = y_train - base_train

theta1 = 0.65
theta2 = -0.25
efeito_ma = np.zeros_like(residuos_base)
for t in range(2, len(residuos_base)):
    efeito_ma[t] = residuos_base[t] + theta1 * residuos_base[t - 1] + theta2 * residuos_base[t - 2]

plt.figure(figsize=(12, 4.8))
plt.plot(x_train, residuos_base, color='#8ce99a', linewidth=2, label='Residuos')
plt.plot(x_train, efeito_ma, color='#ff8fab', linewidth=2.2, label='Combinacao MA dos erros')
plt.axhline(0, color='white', linestyle=':', alpha=0.6)
plt.title('Intuicao da parte MA: erro atual + erros passados')
plt.xlabel('Mes')
plt.ylabel('Erro')
plt.legend()
plt.show()

---

## 4. Previsao com regressao linear

Aqui tratamos o tempo como variavel explicativa. O modelo procura a melhor reta:

$$y_t = a + bt$$

In [ ]:
coef = np.polyfit(x_train, y_train, deg=1)
slope, intercept = coef[0], coef[1]

pred_train_reg = slope * x_train + intercept
pred_test_reg = slope * x_test + intercept

print(f'Inclinação estimada: {slope:.3f}')
print(f'Intercepto estimado: {intercept:.3f}')

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(meses, serie, color='#74c7ec', linewidth=2.2, label='Serie real')
plt.plot(x_train, pred_train_reg, color='#ffd166', linewidth=2.5, label='Ajuste por regressao')
plt.plot(x_test, pred_test_reg, color='#ff8fab', linewidth=2.5, linestyle='--', label='Previsao por regressao')
plt.axvline(train_size - 1, color='white', linestyle=':', alpha=0.7)
plt.title('Regressao linear para prever o futuro')
plt.xlabel('Mes')
plt.ylabel('Usuarios ativos (milhares)')
plt.legend()
plt.show()

### Limite da regressao

Ela enxerga bem a direcao geral da serie, mas ignora memoria curta e estrutura dos erros.

---

## 5. Previsao com ARIMA

Agora juntamos tudo:

- o `I` diferencia a serie
- o `AR` usa valores passados
- o `MA` usa erros passados

Em resumo, **ARIMA e um ARMA aplicado sobre uma serie diferenciada**.

Agora usamos um modelo que considera o passado da propria serie.

Vamos testar um **ARIMA(2, 2, 3)** para capturar dinamica temporal e diferencas sucessivas da serie.

In [ ]:
modelo_arima = ARIMA(y_train, order=(2, 2, 3))
resultado_arima = modelo_arima.fit()

pred_test_arima = resultado_arima.forecast(steps=len(y_test))

print(resultado_arima.summary().tables[1])

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(meses, serie, color='#74c7ec', linewidth=2.2, label='Serie real')
plt.plot(x_test, pred_test_reg, color='#ffd166', linewidth=2.2, linestyle='--', label='Regressao linear')
plt.plot(x_test, pred_test_arima, color='#ff8fab', linewidth=2.5, label='ARIMA(2,2,3)')
plt.axvline(train_size - 1, color='white', linestyle=':', alpha=0.7)
plt.title('Comparando previsoes: regressao vs ARIMA')
plt.xlabel('Mes')
plt.ylabel('Usuarios ativos (milhares)')
plt.legend()
plt.show()

---

## 6. Medindo erro de previsao

Vamos usar o **MAE** (erro absoluto medio) para comparar os modelos.

In [ ]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

mae_reg = mae(y_test, pred_test_reg)
mae_arima = mae(y_test, pred_test_arima)

print(f'MAE regressao linear: {mae_reg:.3f}')
print(f'MAE ARIMA: {mae_arima:.3f}')

In [ ]:
erros_reg = y_test - pred_test_reg
erros_arima = y_test - pred_test_arima

plt.figure(figsize=(12, 4.5))
plt.plot(x_test, erros_reg, color='#ffd166', linewidth=2, label='Erro da regressao')
plt.plot(x_test, erros_arima, color='#8ce99a', linewidth=2, label='Erro do ARIMA')
plt.axhline(0, color='white', linestyle=':', alpha=0.7)
plt.title('Erros residuais nos 12 meses de teste')
plt.xlabel('Mes')
plt.ylabel('Erro')
plt.legend()
plt.show()

### Leitura probabilistica

Os residuos representam a parte que o modelo nao explicou.

Em estatistica aplicada, eles ajudam a responder:
- o erro parece centrado em zero?
- o modelo esta enviesado?
- a incerteza cresce com o horizonte de previsao?

---

## 7. Fazendo uma previsao para frente

Agora vamos ajustar o ARIMA na serie completa e projetar os proximos 12 meses.

In [ ]:
modelo_final = ARIMA(serie, order=(2, 2, 3))
resultado_final = modelo_final.fit()

horizonte = 12
futuro_x = np.arange(len(serie), len(serie) + horizonte)
futuro_y = resultado_final.forecast(steps=horizonte)

plt.figure(figsize=(12, 5))
plt.plot(meses, serie, color='#74c7ec', linewidth=2.2, label='Historico')
plt.plot(futuro_x, futuro_y, color='#ff8fab', linewidth=2.6, label='Previsao futura ARIMA')
plt.axvline(len(serie) - 1, color='white', linestyle=':', alpha=0.7)
plt.title('Tentando prever o futuro com ARIMA')
plt.xlabel('Mes')
plt.ylabel('Usuarios ativos (milhares)')
plt.legend()
plt.show()

---

## 8. Conclusoes

O que vimos nesta aula:

1. Estatistica ajuda a entender a estrutura da serie
2. Probabilidade ajuda a pensar em erro e incerteza
3. Regressao linear e uma boa base para tendencia
4. ARIMA adiciona memoria temporal e costuma melhorar previsao em series dependentes do passado

Em resumo:
- **Regressao** explica a direcao media
- **ARIMA** modela dinamica temporal
- **Graficos lineares** tornam a comparacao intuitiva

Prever o futuro nao e magia. E extrair padroes do passado com cuidado estatistico.